# A Cappella Transcription: Vocal Separation on Cloud GPU

This notebook provides an NVIDIA GPU environment to run the `Phase 2` vocal separation pipeline.

**Hardware & Template Requirements:**
- **VRAM**: A `20-second` audio chunk scales quadratically to require roughly **20-24 GB VRAM** (e.g., L4, RTX 3090/4090, RTX A5000).
- **Pod Template**: Select a template with **PyTorch 2.1+ and CUDA 12.1+** (e.g., the official `RunPod PyTorch 2.1` or `2.2` template). SepACap, STARS, and ROSVOT all rely on modern `torchaudio` features that ship with PyTorch 2.x.

**Future Development Workflow (STARS & ROSVOT):**
- The upcoming Phase 3 Transcription models (STARS and ROSVOT) are also deep learning architectures that convert audio to symbolic MIDI. Running them on Apple Silicon will bottleneck development.
- **Workflow**: Instead of moving files back and forth via Github, deploy a standard SSH Pod on RunPod, connect your local VS Code to it using **VS Code Remote-SSH**, and install the Anthropic/Antigravity extension directly on the remote server. Antigravity can then write code, run terminals, and read the GPU VRAM natively on the cloud instance!

**Instructions:**
1. Clone this repository to your RunPod/Cloud instance.
2. Run the environment setup cells below to download the submodules and install dependencies.
3. Upload your `.wav` or `.mp3` files to the `data/input/` folder.
4. Execute the separator script.

## 1. System Setup

In [ ]:
!sudo apt-get update && sudo apt-get install -y ffmpeg git
!mkdir -p ext
!git clone https://github.com/facebookresearch/demucs ext/demucs
!git clone https://github.com/ETH-DISCO/SepACap ext/SepACap
!git clone https://github.com/a-wei-cat/STARS ext/STARS
!git clone https://github.com/yinshengwang/ROSVOT ext/ROSVOT
!pip install -r requirements.txt
!pip install loguru torchmetrics einops torchaudio huggingface_hub pyyaml soundfile

## 2. Workspace Preparation

In [ ]:
import os

# Ensure we are in the root directory
if not os.path.exists('scripts/run_phase2.py'):
    if os.path.exists('A-Cappella-Transcription/scripts/run_phase2.py'):
        %cd A-Cappella-Transcription
    else:
        print("Please ensure the A-Cappella-Transcription directory is uploaded to this instance!")

# Create data directories for input and output
os.makedirs('data/input', exist_ok=True)
os.makedirs('data/output/prep', exist_ok=True)
os.makedirs('data/output/separation', exist_ok=True)
print("Directory structure prepared. Please map or upload your audio files to 'data/input/' now.")

In [ ]:
import shutil
import os

# Since the repo tracks infer_satb.py locally, we need to inject the wrapper into the newly cloned ext/SepACap.
if os.path.exists('ext/SepACap/infer_satb.py'):
   print('ext/SepACap/infer_satb.py already present!')
else:
   # Because git submodules were removed, you must ensure you have uploaded your local copy of infer_satb.py
   # to the cloud before running Phase 2! This will throw an error if you forget.
   pass

## 3. Run Pipeline
Upload your audio to `data/input/` and upload your modified `infer_satb.py` into `ext/SepACap/` before running this step.

In [ ]:
!PYTHONPATH=. python scripts/run_phase2.py